# **Preprocessing**

In [1]:
import pandas as pd
import numpy as np
import re
from typing import List, Dict, Tuple
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold
from scipy.stats import chi2_contingency, spearmanr

df = pd.read_csv("merged_data.csv")

Helper function - lấy từ file eda

In [2]:
def get_feature_lists(df: pd.DataFrame, target_col: str, id_col: str | None = None) -> tuple[list[str], list[str]]:
    excluded = {target_col}
    if id_col:
        excluded.add(id_col)

    numeric_cols = [col for col in df.select_dtypes(include=np.number).columns if col not in excluded]
    categorical_cols = [col for col in df.columns if col not in excluded and col not in numeric_cols]
    return numeric_cols, categorical_cols


def build_missing_signal_table(df: pd.DataFrame, target_col: str, positive_class) -> pd.DataFrame:
    rows = []
    fraud_mask = df[target_col] == positive_class
    nonfraud_mask = df[target_col] != positive_class

    for col in df.columns:
        if col == target_col:
            continue

        is_missing = df[col].isna()
        overall_missing = is_missing.mean()

        if overall_missing == 0:
            continue

        fraud_missing = is_missing[fraud_mask].mean() if fraud_mask.any() else np.nan
        nonfraud_missing = is_missing[nonfraud_mask].mean() if nonfraud_mask.any() else np.nan
        fraud_rate_if_missing = df.loc[is_missing, target_col].eq(positive_class).mean()
        fraud_rate_if_present = df.loc[~is_missing, target_col].eq(positive_class).mean()

        rows.append(
            {
                "feature": col,
                "overall_missing_pct": overall_missing,
                "fraud_missing_pct": fraud_missing,
                "nonfraud_missing_pct": nonfraud_missing,
                "missing_gap": fraud_missing - nonfraud_missing if pd.notna(fraud_missing) and pd.notna(nonfraud_missing) else np.nan,
                "abs_missing_gap": abs(fraud_missing - nonfraud_missing) if pd.notna(fraud_missing) and pd.notna(nonfraud_missing) else np.nan,
                "fraud_rate_if_missing": fraud_rate_if_missing,
                "fraud_rate_if_present": fraud_rate_if_present,
            }
        )

    result = pd.DataFrame(rows)
    if result.empty:
        return result
    return result.sort_values("abs_missing_gap", ascending=False).reset_index(drop=True)


def feature_family(col_name: str) -> str:
    if col_name in {"TransactionID", "TransactionDT", "TransactionAmt"}:
        return "Transaction"
    if col_name.startswith("card"):
        return "card"
    if col_name.startswith("addr"):
        return "addr"
    if col_name.startswith("dist"):
        return "dist"
    if col_name.startswith("P_email"):
        return "P_emaildomain"
    if col_name.startswith("R_email"):
        return "R_emaildomain"
    if re.fullmatch(r"C\d+", col_name):
        return "C"
    if re.fullmatch(r"D\d+", col_name):
        return "D"
    if re.fullmatch(r"M\d+", col_name):
        return "M"
    if re.fullmatch(r"V\d+", col_name):
        return "V"
    if re.fullmatch(r"id_\d+", col_name):
        return "id"
    if col_name.startswith("Device"):
        return "Device"
    if col_name == "ProductCD":
        return "Product"
    if col_name == "isFraud":
        return "Target"
    return str(col_name).split("_")[0]

## **1. Handle missing value**

- Các cột missing value ~ 99% và không có class signal, drop cột
- Numerical: tạo indicator của các biến có signal mạnh, sau đó fill bằng median và -999
- Cột categorical fill giá trị 'missing' vào NaN

In [3]:
cols_to_drop = ['id_24', 'id_25', 'id_07', 'id_08', 'id_21','id_26','id_22','id_23','id_27']

df.drop(columns=cols_to_drop, inplace=True)

In [4]:
def get_top_missing_features(
    df_train: pd.DataFrame,
    target_col: str = "isFraud",
    top_k: int = 60
) -> Tuple[List[str], Dict, pd.DataFrame]:

    if target_col not in df_train.columns:
        raise ValueError(f"Target column '{target_col}' not found!")

    fraud_mask = df_train[target_col] == 1
    rows = []

    for col in df_train.columns:
        if col == target_col:
            continue

        is_missing = df_train[col].isna()
        missing_pct = is_missing.mean()

        if missing_pct < 0.001:
            continue

        # Missing gap
        fraud_missing_pct = is_missing[fraud_mask].mean()
        nonfraud_missing_pct = is_missing[~fraud_mask].mean()
        abs_gap = abs(fraud_missing_pct - nonfraud_missing_pct)

        # Fraud rate gap
        fraud_rate_missing = df_train.loc[is_missing, target_col].mean() if is_missing.any() else 0.0
        fraud_rate_present = df_train.loc[~is_missing, target_col].mean()
        fraud_rate_gap = abs(fraud_rate_missing - fraud_rate_present)

  
        if pd.api.types.is_numeric_dtype(df_train[col]):
            temp = df_train[[col, target_col]].copy()
            temp[col] = temp[col].fillna(temp[col].median())
            corr = abs(spearmanr(temp[col], temp[target_col], nan_policy='omit')[0])
            if np.isnan(corr):
                corr = 0.0
        else:
            contingency = pd.crosstab(
                df_train[col].fillna("MISSING"), 
                df_train[target_col],
                dropna=False
            )
            if contingency.shape[0] > 1 and contingency.shape[1] > 1:
                chi2, _, _, _ = chi2_contingency(contingency)
                n = df_train.shape[0]
                min_dim = min(contingency.shape) - 1
                corr = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0.0
            else:
                corr = 0.0

        score = (
            0.35 * abs_gap +
            0.30 * fraud_rate_gap +
            0.25 * corr +
            0.10 * missing_pct
        )

        rows.append({
            "feature": col,
            "score": score,
            "abs_gap": abs_gap,
            "fraud_rate_gap": fraud_rate_gap,
            "missing_pct": missing_pct,
            "signal_corr": corr,
            "dtype": str(df_train[col].dtype)
        })

    ranking_df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    top_features = ranking_df.head(top_k)["feature"].tolist()

    impute_stats = {}
    for col in top_features:
        if pd.api.types.is_numeric_dtype(df_train[col]):
            impute_stats[col] = df_train[col].median()
        else:
            impute_stats[col] = "MISSING"

    return top_features, impute_stats, ranking_df


def preprocess_missing(
    df: pd.DataFrame,
    top_features: List[str],
    impute_stats: Dict,
    target_col: str = "isFraud"
) -> pd.DataFrame:

    df = df.copy()

    indicator_list = []
    for col in top_features:
        if col not in df.columns:
            continue
        indicator = df[col].isna().astype("int8")
        indicator.name = f"{col}_isna"
        indicator_list.append(indicator)

    if indicator_list:
        indicators_df = pd.concat(indicator_list, axis=1)
        df = pd.concat([df, indicators_df], axis=1)

    for col in top_features:
        if col not in df.columns:
            continue

        if pd.api.types.is_numeric_dtype(df[col]):
            fill_val = impute_stats.get(col, -999)
            df[col] = df[col].fillna(fill_val)
        else:
            df[col] = df[col].astype("string").fillna("MISSING").astype("category")

    return df

In [5]:
top_features, impute_stats, ranking_df = get_top_missing_features(
    df, 
    target_col="isFraud", 
    top_k=80
)

df_clean = preprocess_missing(
    df, 
    top_features, 
    impute_stats
)

print(f"Số cột ban đầu: {df.shape[1]} -> Số cột sau xử lý missing: {df_clean.shape[1]}")

Số cột ban đầu: 427 -> Số cột sau xử lý missing: 507


## **Prunning**
Prune các cột tương quan cao, ưu tiên giữ lại feature có signal mạnh với target.

In [6]:
def prune_highly_correlated_features(
    df: pd.DataFrame,
    target_col: str = "isFraud",
    corr_threshold: float = 0.85,
    v_only: bool = True
) -> Tuple[pd.DataFrame, List[str], pd.DataFrame]:

    df = df.copy()

    if v_only:
        numeric_cols = [col for col in df.columns if col.startswith("V") and col != target_col]
    else:
        numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
        if target_col in numeric_cols:
            numeric_cols.remove(target_col)
    
    if len(numeric_cols) < 2:
        return df, [], pd.DataFrame()

    corr_matrix = df[numeric_cols].corr(method="pearson").abs()
    
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_pairs = upper.stack().reset_index()
    high_corr_pairs.columns = ["feature_1", "feature_2", "corr"]
    high_corr_pairs = high_corr_pairs[high_corr_pairs["corr"] >= corr_threshold]
    
    target_corr = {}
    for col in numeric_cols:
        target_corr[col] = abs(df[col].corr(df[target_col], method="spearman"))
    

    to_drop = set()
    processed = set()
    
    for _, row in high_corr_pairs.sort_values("corr", ascending=False).iterrows():
        f1, f2 = row["feature_1"], row["feature_2"]
        if f1 in to_drop or f2 in to_drop:
            continue
        
        # So sánh signal với target
        if target_corr.get(f1, 0) >= target_corr.get(f2, 0):
            to_drop.add(f2)
        else:
            to_drop.add(f1)
    
    df_pruned = df.drop(columns=list(to_drop))
    kept_features = [col for col in numeric_cols if col not in to_drop]
    
    print(f"Pruned {len(to_drop)} cột tương quan cao.")
    print(f"   - Giữ lại: {len(kept_features)} cột V*")
    print(f"   - Threshold: {corr_threshold}")

    summary = pd.DataFrame({
        "dropped_feature": list(to_drop),
        "reason": "high_correlation"
    })
    
    return df_pruned, kept_features, summary

In [7]:
df_pruned, kept_v_features, prune_summary = prune_highly_correlated_features(
    df_clean,
    target_col="isFraud",
    corr_threshold=0.8,      
    v_only=True               
)

print(f"Final shape sau prune: {df_pruned.shape}")

Pruned 289 cột tương quan cao.
   - Giữ lại: 110 cột V*
   - Threshold: 0.8
Final shape sau prune: (590540, 218)
